In [58]:
using JLD2
using Flux
using ONNXNaiveNASflux
using Random
using LinearAlgebra
using ModelVerification
using Cersyve

In [60]:
task = RobotDog
task_name = "robot_dog"
version = "finetune"

value_hidden_sizes = [32, 32]
dynamics_hidden_sizes = [32, 32]
constraint_hidden_sizes = [16]
data_path = joinpath(@__DIR__, "../data/$(task_name)_data.jld2")
model_dir = joinpath(@__DIR__, "../model/$task_name/")
onnx_dir = joinpath(@__DIR__, "../onnx/")

V_model = Cersyve.create_mlp(task.x_dim, 1, value_hidden_sizes)
Flux.loadmodel!(V_model, JLD2.load(joinpath(model_dir, "V_$(version).jld2"), "state"))

# data = JLD2.load(data_path)["data"]
# f_model = Cersyve.create_mlp(task.x_dim + task.u_dim, task.x_dim, dynamics_hidden_sizes)
# Flux.loadmodel!(f_model, JLD2.load(joinpath(model_dir, "f.jld2"), "state"))
# f_pi_model = Cersyve.create_closed_loop_dynamics_model(
#     f_model, task.pi_model, data, task.x_low, task.x_high, task.u_dim)

# f_pi_model = task.f_pi_model

# for RobotDog only
f_model = ModelVerification.build_flux_model(task.dynamics_path)
f_pi_model = Chain(Parallel(+,
    Dense(Matrix{Float32}(I(task.x_dim))),
    f_model,
))

h_model = Cersyve.create_mlp(task.x_dim, 1, constraint_hidden_sizes)
Flux.loadmodel!(h_model, JLD2.load(joinpath(model_dir, "h.jld2"), "state"))

# h_model = task.h_model

con_model = Cersyve.create_value_constraint_model(V_model, h_model)
inv_model = Cersyve.create_value_next_value_model(V_model, f_pi_model)
ONNXNaiveNASflux.save(joinpath(onnx_dir, "$(task_name)_$(version)_con.onnx"), con_model, (task.x_dim, 1))
ONNXNaiveNASflux.save(joinpath(onnx_dir, "$(task_name)_$(version)_inv.onnx"), inv_model, (task.x_dim, 1))

17995